# Predictive Maintenance: EDA & Modeling
AI4I 2020 dataset: 10,000 machines, about 3.4% of which failed. This notebook walks through Steps 2–8 of the guide and goes further. It reuses the functions in `train.py` and `predict.py`, so the notebook and the production pipeline always match.

In [ ]:
import os, sys
os.chdir('..'); sys.path.insert(0, '.')
import pandas as pd
import train, predict
from IPython.display import Image, Markdown, display

## Step 2: Load data

In [ ]:
df = train.load_data()
df.describe()

## Step 3: EDA
The histograms and correlations include three engineered features: power (torque × speed), the process–air temperature gap, and strain (wear × torque).

In [ ]:
train.eda(df)
display(Image('reports/distributions.png'), Image('reports/correlations.png'))

## Step 4: Preprocessing
* The failure-mode flags (TWF, HDF, PWF, OSF, RNF) are dropped because they **leak the target**.
* The data is split **before** any undersampling, so the test set keeps the real 3.4% failure rate.

In [ ]:
X_train, X_test, y_train, y_test, X_bal, y_bal = train.preprocess(df)

## Step 5: The guide's 3 models (trained on undersampled data)

In [ ]:
fitted, rows = {}, []
for name, model in train.baseline_models().items():
    model.fit(X_bal, y_bal)
    fitted[name] = model
    rows.append({'model': name, **{k: v for k, v in train.test_metrics(model, X_test, y_test, 0.5).items() if k != 'confusion_matrix'}})
pd.DataFrame(rows).set_index('model').round(3)

## Beyond the guide: tuned, class-weighted XGBoost
This model trains on **all** training rows and handles the imbalance with `scale_pos_weight`, instead of throwing away 94% of the healthy machines. Hyperparameters are tuned for average precision. The alert threshold is then chosen by cross-validation on the training data only, to reach recall ≥ 0.85.

In [ ]:
name = 'XGBoost tuned (class-weighted)'
tuned, best_params, spw = train.tune_xgboost(X_train, y_train)
fitted[name] = tuned
threshold = train.choose_threshold(tuned, X_train, y_train)
final = train.test_metrics(tuned, X_test, y_test, threshold)
final

## Step 6: Evaluation & explanation

In [ ]:
perm = train.plots(fitted, name, X_test, y_test)
for img in ['roc_curve', 'pr_curve', 'permutation_importance', 'feature_importance']:
    display(Image(f'reports/{img}.png'))
perm

## Steps 7–8: Score new machines with the saved model
(`train.py` saves the model; `predict.py` adds the engineered features and applies the alert threshold.)

In [ ]:
test = pd.read_csv('reports/test_set.csv')
sample = pd.concat([test[test['Machine failure'] == 1].head(3), test[test['Machine failure'] == 0].head(3)])
sample[predict.RAW_FEATURES + ['Machine failure']].join(predict.predict_failure_risk(sample))